In [1]:
# Imports

import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Definir rutas

DATA_DIR     = "/beegfs/home/iruizdealda/HCC_singlecell_project/data"
RESULTS_DIR  = "/beegfs/home/iruizdealda/HCC_singlecell_project/results"
FIGURES_DIR  = "/beegfs/home/iruizdealda/HCC_singlecell_project/figures"
DATASET_PATH = f"{DATA_DIR}/Response_AtezoBev_HCC"

In [3]:
# Comprobar archivos disponibles

os.listdir(DATASET_PATH)

['matrix.mtx.gz', 'meta_data.txt', 'barcodes.tsv', 'features.tsv']

In [5]:
import gzip
import shutil
from scipy.io import mmread

# Descomprimir los archivos .gz primero
for fname in ["matrix.mtx.gz", "barcodes.tsv.gz", "features.tsv.gz"]:
    src = os.path.join(DATASET_PATH, fname)
    dst = os.path.join(DATASET_PATH, fname.replace(".gz", ""))
    if not os.path.exists(dst):
        print(f"Descomprimiendo {fname}...")
        with gzip.open(src, 'rb') as f_in, open(dst, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        print(f"  -> {dst}")
    else:
        print(f"Ya existe: {dst}")

# Leer matriz descomprimida
X = mmread(os.path.join(DATASET_PATH, "matrix.mtx")).tocsr().T

# Leer barcodes
barcodes = pd.read_csv(
    os.path.join(DATASET_PATH, "barcodes.tsv"),
    header=None, sep="\t"
)[0].tolist()

# Leer features
features = pd.read_csv(
    os.path.join(DATASET_PATH, "features.tsv"),
    header=None, sep="\t"
)

# Usar columna 1 (nombre gen) si existe, si no columna 0
gene_col = 1 if features.shape[1] > 1 else 0
print(f"Features: {features.shape[1]} columna(s) → usando columna {gene_col}")

# Crear AnnData
adata = sc.AnnData(
    X=X,
    obs=pd.DataFrame(index=barcodes),
    var=pd.DataFrame(index=features[gene_col].astype(str).tolist())
)

print(adata)
print(adata.shape)

Ya existe: /beegfs/home/iruizdealda/HCC_singlecell_project/data/Response_AtezoBev_HCC/matrix.mtx
Ya existe: /beegfs/home/iruizdealda/HCC_singlecell_project/data/Response_AtezoBev_HCC/barcodes.tsv
Ya existe: /beegfs/home/iruizdealda/HCC_singlecell_project/data/Response_AtezoBev_HCC/features.tsv
Features: 1 columna(s) → usando columna 0
AnnData object with n_obs × n_vars = 97947 × 36601
(97947, 36601)


Se cargó el dataset Response_AtezoBev_HCC desde archivos comprimidos (.gz).
Los archivos matrix.mtx.gz, barcodes.tsv.gz y features.tsv.gz se leyeron
directamente sin necesidad de descomprimirlos previamente.
La matriz se transpuso para obtener el formato estándar células x genes.
El resultado es un objeto AnnData donde las filas son células y las columnas son genes.

In [6]:
# Hacer únicos los nombres de los genes
#
# Es habitual que algunos genes aparezcan duplicados en el archivo
# de features. Se aplica var_names_make_unique() para evitar
# problemas en pasos posteriores del análisis.

adata.var_names_make_unique()

print("Genes únicos:", adata.n_vars)
print("Células:", adata.n_obs)

Genes únicos: 36601
Células: 97947


In [7]:
# Leer y explorar el archivo de metadata
#
# A diferencia de GSE125449, este dataset incluye un archivo meta_data.txt
# con información clínica de cada célula: Patient_ID, Response al tratamiento,
# Treatment y Week de recogida de la muestra.
# El índice del meta_data (Tumour_ID) contiene el barcode de cada célula,
# que usaremos para hacer el join con el AnnData.

meta = pd.read_csv(
    os.path.join(DATASET_PATH, "meta_data.txt"),
    sep="\t",
    index_col=0
)

print("Shape metadata:", meta.shape)
print("\nColumnas:", meta.columns.tolist())
print("\nPrimeras filas:")
meta.head()

Shape metadata: (97947, 5)

Columnas: ['Tumour_ID', 'Patient_ID', 'Response', 'Treatment', 'Week']

Primeras filas:


,Tumour_ID,Patient_ID,Response,Treatment,Week
AAACCTGCACAGACTT-1_tumour_1,tumour_1,pat24,Responder,atezo+bev,W0
AAACCTGCACGAAACG-1_tumour_1,tumour_1,pat24,Responder,atezo+bev,W0
AAACCTGTCGGAAATA-1_tumour_1,tumour_1,pat24,Responder,atezo+bev,W0
AAACGGGCACATGGGA-1_tumour_1,tumour_1,pat24,Responder,atezo+bev,W0
AAACGGGCAGTGGAGT-1_tumour_1,tumour_1,pat24,Responder,atezo+bev,W0


In [8]:
# Explorar valores únicos de las columnas clínicas

print("Patient_ID únicos:", meta["Patient_ID"].nunique())
print(meta["Patient_ID"].value_counts())

print("\nResponse:")
print(meta["Response"].value_counts())

print("\nTreatment:")
print(meta["Treatment"].value_counts())

print("\nWeek:")
print(meta["Week"].value_counts())

Patient_ID únicos: 38
Patient_ID
pat13    7815
pat37    7311
pat34    6838
pat36    4467
pat3     4334
pat2     4037
pat30    3881
pat16    3739
pat4     3715
pat11    3628
pat9     3276
pat14    3086
pat32    3033
pat35    3027
pat28    2982
pat23    2981
pat10    2722
pat6     2576
pat38    2511
pat29    2255
pat17    2226
pat7     2106
pat22    1854
pat31    1809
pat24    1720
pat19    1680
pat27    1520
pat8     1323
pat12    1312
pat33    1176
pat26    1121
pat18     708
pat21     440
pat15     302
pat20     168
pat1      136
pat5       74
pat25      58
Name: count, dtype: int64

Response:
Response
Responder             61947
NonResponder          22444
DeathBeforeImaging     7822
Name: count, dtype: int64

Treatment:
Treatment
atezo+bev     46023
ICP           22939
TKI           15436
atezo+cabo     7815
Name: count, dtype: int64

Week:
Week
W0    97947
Name: count, dtype: int64


El archivo meta_data.txt contiene información clínica por célula:
- Patient_ID: identificador del tumor de origen de cada célula
- Response: si el paciente respondió o no al tratamiento (Responder / Non-responder)
- Treatment: tratamiento recibido (atezolizumab + bevacizumab)
- Week: semana de recogida de la muestra (W0 = baseline pre-tratamiento)

Esta información clínica es la que diferencia este dataset de los demás
y permite el análisis comparativo Responders vs Non-responders.

In [9]:
# Comprobar solapamiento entre barcodes de la matriz y del meta_data
#
# Es importante verificar que los índices del meta_data coinciden
# con los barcodes del AnnData antes de hacer el join.

comunes = adata.obs_names.intersection(meta.index)
print(f"Células en AnnData:  {adata.n_obs}")
print(f"Células en metadata: {len(meta)}")
print(f"Barcodes comunes:    {len(comunes)}")

if len(comunes) == 0:
    print("\n[AVISO] No hay barcodes comunes. Revisar formato del índice.")
    print("Ejemplo barcode AnnData:", adata.obs_names[0])
    print("Ejemplo barcode meta:   ", meta.index[0])

Células en AnnData:  97947
Células en metadata: 97947
Barcodes comunes:    97947


In [10]:
# Añadir metadata clínica al obs del AnnData
#
# Se hace un join entre el obs del AnnData y el DataFrame de metadata,
# usando el barcode como clave. Las columnas de metadata quedan
# disponibles en adata.obs para colorear UMAPs, filtrar células, etc.

adata.obs = adata.obs.join(meta, how="left")

print("Columnas en adata.obs tras el join:")
print(adata.obs.columns.tolist())

print("\nCélulas con metadata asignada:")
print(adata.obs["Response"].notna().sum(), "/", adata.n_obs)

Columnas en adata.obs tras el join:
['Tumour_ID', 'Patient_ID', 'Response', 'Treatment', 'Week']

Células con metadata asignada:
92213 / 97947


Se añadieron las columnas clínicas al objeto AnnData mediante un join
usando el barcode (Tumour_ID) como clave de unión.
A diferencia de GSE125449, donde solo se añadía la columna 'batch',
aquí el obs queda enriquecido con información clínica real del paper:
Patient_ID, Response, Treatment y Week.

In [11]:
# Añadir columna batch para identificar el dataset en análisis integrativos

adata.obs["batch"] = "Response_AtezoBev_HCC"

print(adata)
print("\nResumen obs:")
print(adata.obs.head())

AnnData object with n_obs × n_vars = 97947 × 36601
    obs: 'Tumour_ID', 'Patient_ID', 'Response', 'Treatment', 'Week', 'batch'

Resumen obs:
                            Tumour_ID Patient_ID   Response  Treatment Week  \
AAACCTGCACAGACTT-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0   
AAACCTGCACGAAACG-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0   
AAACCTGTCGGAAATA-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0   
AAACGGGCACATGGGA-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0   
AAACGGGCAGTGGAGT-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0   

                                             batch  
AAACCTGCACAGACTT-1_tumour_1  Response_AtezoBev_HCC  
AAACCTGCACGAAACG-1_tumour_1  Response_AtezoBev_HCC  
AAACCTGTCGGAAATA-1_tumour_1  Response_AtezoBev_HCC  
AAACGGGCACATGGGA-1_tumour_1  Response_AtezoBev_HCC  
AAACGGGCAGTGGAGT-1_tumour_1  Response_AtezoBev_HCC  


In [12]:
# Guardar el AnnData procesado

out_path = f"{DATA_DIR}/adata_Response_AtezoBev_HCC_raw.h5ad"
adata.write(out_path)
print(f"Guardado en: {out_path}")

Guardado en: /beegfs/home/iruizdealda/HCC_singlecell_project/data/adata_Response_AtezoBev_HCC_raw.h5ad
